# Notebook 17 — Constraint-gated stability and invariance

**Repo path:** `control-stack/17_constraint_gated_stability_and_invariance.ipynb`

This notebook turns the control-stack result from an empirical method into a compact stability principle.

**same math · lifted clarity**

## Core claim

A control update constrained by the 45° CGCS gate keeps phase-error accumulation bounded under noise bursts and model mismatch.

```text
adaptive + unconstrained → drift accumulation
adaptive + CGCS-constrained → bounded update propagation
```


In [ ]:
# Notebook 17 — setup

import os
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "17"
SLUG = "constraint_gated_stability_and_invariance"
TITLE = "Constraint-gated stability and invariance"

FIG_DIR = f"figures/{SLUG}"
RESULTS_DIR = "results"
DOCS_DIR = "docs"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DOCS_DIR, exist_ok=True)

np.random.seed(9423)

THRESHOLD = 1 / np.sqrt(1**2 + 1**2)
EPS = 1e-12

print(f"45° threshold = 1 / sqrt(1² + 1²) = {THRESHOLD:.6f}")
print(f"Figures -> {FIG_DIR}")
print(f"Results -> {RESULTS_DIR}")
print(f"Docs -> {DOCS_DIR}")


## 1. Formal stability object

Define response alignment by cosine similarity. The constraint gate is active when response alignment stays above the 45° threshold:

```text
cosθ ≥ 1 / sqrt(1² + 1²)
```

The constrained update rule is:

```text
x_hat[t+1] = x_hat[t] + gate[t] * K[t] * innovation[t]
```

where `gate[t] = 1` only when the proposed update keeps response alignment inside the CGCS-safe region.


In [ ]:
# Synthetic calibration stream with noise bursts and model mismatch windows

n = 90
t = np.arange(n)
pulse = np.linspace(0, 10, 300)

noise_burst_windows = [(30, 42), (58, 70)]
model_mismatch_windows = [(48, 56), (72, 82)]

def in_windows(i, windows):
    return any(start <= i < end for start, end in windows)

def shade_windows(ax):
    first_noise = True
    first_mismatch = True
    for start, end in noise_burst_windows:
        ax.axvspan(start, end, alpha=0.17, label="noise burst" if first_noise else None)
        first_noise = False
    for start, end in model_mismatch_windows:
        ax.axvspan(start, end, alpha=0.10, label="model mismatch" if first_mismatch else None)
        first_mismatch = False

omega_true = (
    0.045*np.sin(2*np.pi*t/22)
    + 0.020*np.sin(2*np.pi*t/8)
    + 0.00035*t
)

B_true = (
    0.006*np.sin(2*np.pi*t/14)
    + 0.00075*t
    + 0.018/(1 + np.exp(-(t-50)/4))
)

omega_noise = np.random.normal(0, 0.005, n)
B_noise = np.random.normal(0, 0.004, n)

omega_meas = omega_true + omega_noise
B_meas = B_true + B_noise

for i in range(n):
    if in_windows(i, noise_burst_windows):
        omega_meas[i] += np.random.normal(0, 0.040)
        B_meas[i] += np.random.normal(0, 0.028)
    if in_windows(i, model_mismatch_windows):
        omega_meas[i] += 0.055*np.sin(0.8*i)
        B_meas[i] += 0.032*np.cos(0.6*i)

# A few deterministic outliers
for i, spike in [(16, 0.16), (46, -0.18), (63, 0.12), (78, 0.09)]:
    omega_meas[i] += spike
    B_meas[i] -= 0.55*spike

print("Synthetic stream ready.")
print("noise_burst_windows =", noise_burst_windows)
print("model_mismatch_windows =", model_mismatch_windows)


## 2. Response model and CGCS metrics

The response model is intentionally small: each calibration block produces a target response curve, and each control policy creates a corrected response. The notebook compares response RMSE, cosine similarity, threshold margin, phase-error integral, update acceptance rate, and boundedness / invariance margin.


In [ ]:
# Response and metric utilities

def response_curve(omega, B, mismatch=0.0):
    phase = (1.05 + omega) * pulse + 0.25*B + mismatch
    amp = 0.47 + 0.08*np.tanh(4*B)
    baseline = 0.52 + 0.03*np.sin(B)
    y = baseline + amp*np.sin(phase)**2 - 0.04*np.cos(0.5*pulse + omega)
    return np.clip(y, 0, 1.05)

target_curves = []
measured_curves = []
for i in range(n):
    mismatch = 0.25 if in_windows(i, model_mismatch_windows) else 0.0
    target_curves.append(response_curve(omega_true[i], B_true[i], 0.0))
    measured_curves.append(response_curve(omega_meas[i], B_meas[i], mismatch))

target_curves = np.array(target_curves)
measured_curves = np.array(measured_curves)

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b))**2)))

def cosine_similarity(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a) * np.linalg.norm(b)) + EPS))

def phase_error(cos_values):
    cos_values = np.asarray(cos_values)
    return np.maximum(0, 1 - cos_values)

def threshold_margin(cos_values):
    return np.asarray(cos_values) - THRESHOLD

def moving_average(x, window=5):
    out = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        lo = max(0, i-window+1)
        out[i] = np.mean(x[lo:i+1])
    return out

def joint_kalman_like(omega_z, B_z, coupling=0.15, q=1.8e-4, r=7e-4):
    omega = np.zeros_like(omega_z, dtype=float)
    B = np.zeros_like(B_z, dtype=float)
    omega[0], B[0] = omega_z[0], B_z[0]
    for i in range(1, len(omega_z)):
        pred_o = omega[i-1] + coupling*(B_z[i-1] - B[i-1])
        pred_b = B[i-1] + 0.55*coupling*(omega_z[i-1] - omega[i-1])
        gain = q / (q + r)
        omega[i] = pred_o + gain*(omega_z[i] - pred_o)
        B[i] = pred_b + gain*(B_z[i] - pred_b)
    return omega, B

def robust_gate_filter(z, gate=0.075, q=1.6e-4, r=7e-4):
    x = np.zeros_like(z, dtype=float)
    p = 1.0
    x[0] = z[0]
    rejected = []
    for i in range(1, len(z)):
        pred = x[i-1]
        innov = z[i] - pred
        if abs(innov) <= gate:
            p = p + q
            k = p / (p + r)
            x[i] = pred + k*innov
            p = (1-k)*p
        else:
            x[i] = pred
            rejected.append(i)
    return x, np.array(rejected)

def proposed_response(prev_o, prev_b, new_o, new_b, i):
    mismatch = 0.25 if in_windows(i, model_mismatch_windows) else 0.0
    return response_curve(new_o, new_b, mismatch)

def cgcs_constrained_filter(omega_z, B_z, base_gain=0.26, recovery_gain=0.12, gate_floor=0.03):
    omega = np.zeros_like(omega_z, dtype=float)
    B = np.zeros_like(B_z, dtype=float)
    omega[0], B[0] = omega_z[0], B_z[0]
    accepted = np.zeros(len(omega_z), dtype=int)
    gate_trace = np.zeros(len(omega_z), dtype=float)
    invariant_margin = np.zeros(len(omega_z), dtype=float)

    for i in range(1, len(omega_z)):
        current_curve = response_curve(omega[i-1], B[i-1])
        target_curve = target_curves[i]
        current_cos = cosine_similarity(current_curve, target_curve)
        margin = current_cos - THRESHOLD

        gain = base_gain if margin >= 0 else recovery_gain
        local_gate = max(gate_floor, min(0.10, 0.040 + 0.18*max(margin, 0)))
        gate_trace[i] = local_gate

        pred_o = omega[i-1] + gain*(omega_z[i] - omega[i-1])
        pred_b = B[i-1] + 0.75*gain*(B_z[i] - B[i-1])

        proposal_curve = proposed_response(omega[i-1], B[i-1], pred_o, pred_b, i)
        proposal_cos = cosine_similarity(proposal_curve, target_curve)
        proposed_step = np.sqrt((pred_o-omega[i-1])**2 + (pred_b-B[i-1])**2)

        if (proposal_cos >= THRESHOLD) and (proposed_step <= local_gate):
            omega[i] = pred_o
            B[i] = pred_b
            accepted[i] = 1
        else:
            omega[i] = omega[i-1] + recovery_gain*(omega_z[i] - omega[i-1])
            B[i] = B[i-1] + 0.55*recovery_gain*(B_z[i] - B[i-1])
            accepted[i] = 0

        out_curve = response_curve(omega[i], B[i])
        invariant_margin[i] = cosine_similarity(out_curve, target_curve) - THRESHOLD

    return omega, B, accepted, gate_trace, invariant_margin

def constrained_mpc_lite(omega_z, B_z, command_radius=0.035):
    omega = np.zeros_like(omega_z, dtype=float)
    B = np.zeros_like(B_z, dtype=float)
    omega[0], B[0] = omega_z[0], B_z[0]
    for i in range(1, len(omega_z)):
        d_o = omega_z[i] - omega[i-1]
        d_b = B_z[i] - B[i-1]
        step_norm = np.sqrt(d_o**2 + d_b**2) + EPS
        scale = min(1.0, command_radius / step_norm)
        omega[i] = omega[i-1] + scale*d_o
        B[i] = B[i-1] + scale*d_b
    return omega, B


## 3. Policy set

Notebook 17 keeps the policy set small and formal:

- `none`
- `moving_average`
- `joint_kalman`
- `robust_gated_kalman`
- `constrained_mpc`
- `cgcs_invariance_control`
- `oracle`

The new policy is not meant as a final controller. It demonstrates the update rule:

```text
accept update only when proposed response remains inside the CGCS-safe region
```


In [ ]:
# Build policy estimates

policies = {}

policies["none"] = (omega_meas.copy(), B_meas.copy())
policies["moving_average"] = (moving_average(omega_meas, 7), moving_average(B_meas, 7))
policies["joint_kalman"] = joint_kalman_like(omega_meas, B_meas, coupling=0.18)
robust_o, robust_reject_o = robust_gate_filter(omega_meas, gate=0.075)
robust_b, robust_reject_b = robust_gate_filter(B_meas, gate=0.055)
policies["robust_gated_kalman"] = (robust_o, robust_b)
policies["constrained_mpc"] = constrained_mpc_lite(omega_meas, B_meas, command_radius=0.038)

cgcs_o, cgcs_b, cgcs_accept, cgcs_gate_trace, cgcs_invariant_margin = cgcs_constrained_filter(
    omega_meas, B_meas, base_gain=0.24, recovery_gain=0.10, gate_floor=0.028
)
policies["cgcs_invariance_control"] = (cgcs_o, cgcs_b)

policies["oracle"] = (omega_true.copy(), B_true.copy())

policy_curves = {}
rmse_by_policy = {}
cos_by_policy = {}
margin_by_policy = {}
phase_error_by_policy = {}

for name, (omega_hat, B_hat) in policies.items():
    curves = np.array([response_curve(omega_hat[i], B_hat[i]) for i in range(n)])
    policy_curves[name] = curves
    rmse_by_policy[name] = np.array([rmse(curves[i], target_curves[i]) for i in range(n)])
    cos_by_policy[name] = np.array([cosine_similarity(curves[i], target_curves[i]) for i in range(n)])
    margin_by_policy[name] = threshold_margin(cos_by_policy[name])
    phase_error_by_policy[name] = phase_error(cos_by_policy[name])

rows = []
for name in policies:
    rows.append({
        "policy": name,
        "mean_rmse": float(np.mean(rmse_by_policy[name])),
        "max_rmse": float(np.max(rmse_by_policy[name])),
        "mean_cosine": float(np.mean(cos_by_policy[name])),
        "min_cosine": float(np.min(cos_by_policy[name])),
        "blocks_below_threshold": int(np.sum(cos_by_policy[name] < THRESHOLD)),
        "phase_error_integral": float(np.sum(phase_error_by_policy[name])),
        "mean_margin": float(np.mean(margin_by_policy[name])),
        "min_margin": float(np.min(margin_by_policy[name])),
    })

summary_df = pd.DataFrame(rows).sort_values(["blocks_below_threshold", "mean_rmse"]).reset_index(drop=True)
summary_df


## 4. Phase-lock stability and threshold margin

These are the two core CGCS plots: cosine similarity shows phase-lock stability directly, and threshold margin shows distance from failure boundary.


In [ ]:
plt.figure(figsize=(15, 6))
ax = plt.gca()
shade_windows(ax)
plt.axhline(THRESHOLD, linestyle="--", label="45° threshold")
for name in policies:
    plt.plot(t, cos_by_policy[name], "o-", label=name, alpha=0.90)
plt.title("Constraint-gated stability: CGCS phase-lock stability")
plt.xlabel("calibration block")
plt.ylabel("cosine similarity vs target")
plt.ylim(0.55, 1.02)
plt.legend(ncol=3, fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_cgcs_stability_comparison.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
plt.figure(figsize=(15, 6))
ax = plt.gca()
shade_windows(ax)
plt.axhline(0, linestyle="--", label="threshold margin = 0")
for name in policies:
    if name == "oracle":
        continue
    plt.plot(t, margin_by_policy[name], "o-", label=name, alpha=0.90)
plt.title("Constraint-gated stability: threshold margin")
plt.xlabel("calibration block")
plt.ylabel("CGCS margin above threshold")
plt.legend(ncol=3, fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_threshold_margin.png", dpi=220, bbox_inches="tight")
plt.show()


## 5. Boundedness and invariance diagnostics

The invariance check asks:

```text
Does the controlled response remain inside a bounded CGCS-safe region?
```

This is not a closed-form proof. It is a numerical invariance diagnostic: accepted updates keep the response aligned, and rejected updates prevent misaligned jumps from accumulating.


In [ ]:
# Boundedness / invariance diagnostics

invariance_rows = []
for name in policies:
    cos_vals = cos_by_policy[name]
    margins = margin_by_policy[name]
    rmse_vals = rmse_by_policy[name]
    invariance_rows.append({
        "policy": name,
        "max_phase_error": float(np.max(phase_error_by_policy[name])),
        "phase_error_integral": float(np.sum(phase_error_by_policy[name])),
        "min_threshold_margin": float(np.min(margins)),
        "violation_rate": float(np.mean(cos_vals < THRESHOLD)),
        "bounded_rmse_radius": float(np.quantile(rmse_vals, 0.95)),
    })

invariance_df = pd.DataFrame(invariance_rows).sort_values(["violation_rate", "phase_error_integral"])
invariance_df


In [ ]:
phase_df = invariance_df.sort_values("phase_error_integral")
plt.figure(figsize=(12, 6))
plt.bar(phase_df["policy"], phase_df["phase_error_integral"])
plt.title("Constraint-gated stability: phase-error integral")
plt.ylabel("Σ max(0, 1 - cosine)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_phase_error_integral.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
bound_df = invariance_df.sort_values("bounded_rmse_radius")
plt.figure(figsize=(12, 6))
plt.bar(bound_df["policy"], bound_df["bounded_rmse_radius"])
plt.title("Constraint-gated stability: bounded RMSE radius")
plt.ylabel("95th percentile response RMSE")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_bounded_rmse_radius.png", dpi=220, bbox_inches="tight")
plt.show()


## 6. Control law trace

The gate trace shows the controller changing its update radius rather than blindly increasing update strength. This is the operational difference between adaptive control and CGCS-constrained adaptive control.


In [ ]:
plt.figure(figsize=(15, 5))
ax = plt.gca()
shade_windows(ax)
plt.plot(t, cgcs_gate_trace, label="CGCS adaptive update radius")
plt.axhline(0.075, linestyle="--", label="fixed robust Ω gate")
plt.title("Constraint-gated stability: update-gate trace")
plt.xlabel("calibration block")
plt.ylabel("allowed update radius")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_update_gate_trace.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
plt.figure(figsize=(15, 4))
reject_blocks = np.where(cgcs_accept == 0)[0]
plt.scatter(reject_blocks, np.zeros_like(reject_blocks), marker="x", s=90, label="CGCS rejected update")
plt.yticks([0], ["cgcs_invariance_control"])
plt.title("Constraint-gated stability: rejected update events")
plt.xlabel("calibration block")
plt.ylabel("policy")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_rejection_events.png", dpi=220, bbox_inches="tight")
plt.show()


## 7. Response-level ranking

The ranking plot keeps Notebook 17 compatible with Notebooks 8–16: mean RMSE and max RMSE show practical response-level performance, while the CGCS diagnostics explain why unstable policies fail.


In [ ]:
rank_df = summary_df.sort_values("mean_rmse")
x = np.arange(len(rank_df))
width = 0.38

plt.figure(figsize=(13, 6))
plt.bar(x - width/2, rank_df["mean_rmse"], width, label="mean RMSE")
plt.bar(x + width/2, rank_df["max_rmse"], width, label="max RMSE")
plt.xticks(x, rank_df["policy"], rotation=25, ha="right")
plt.title("Constraint-gated stability: policy ranking")
plt.ylabel("response RMSE")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_policy_ranking_summary.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
plt.figure(figsize=(15, 6))
ax = plt.gca()
shade_windows(ax)
for name in policies:
    if name == "oracle":
        continue
    plt.plot(t, rmse_by_policy[name], "o-", label=name, alpha=0.85)
plt.plot(t, rmse_by_policy["oracle"], "o-", label="oracle", linewidth=2)
plt.title("Constraint-gated stability: response RMSE comparison")
plt.xlabel("calibration block")
plt.ylabel("RMSE vs target response")
plt.legend(ncol=3, fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_response_rmse_comparison.png", dpi=220, bbox_inches="tight")
plt.show()


## 8. Failure-event map

The failure-event map turns the threshold rule into a direct diagnostic:

```text
yellow = below 45° threshold
dark = inside CGCS-safe region
```


In [ ]:
policy_order = list(summary_df["policy"])
failure_matrix = np.vstack([(cos_by_policy[name] < THRESHOLD).astype(int) for name in policy_order])

plt.figure(figsize=(15, 6))
plt.imshow(failure_matrix, aspect="auto", interpolation="nearest")
plt.yticks(np.arange(len(policy_order)), policy_order)
plt.xlabel("calibration block")
plt.ylabel("policy")
plt.title("Constraint-gated stability: failure-event map")
cbar = plt.colorbar()
cbar.set_label("failure: cosθ < threshold")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_failure_event_map.png", dpi=220, bbox_inches="tight")
plt.show()


## 9. Tracking and worst-case response

These plots keep the formal notebook grounded in control-stack behavior: Ω tracking, B tracking, and a worst-case block comparison.


In [ ]:
plt.figure(figsize=(15, 6))
ax = plt.gca()
shade_windows(ax)
plt.plot(t, omega_true, label="true Ω drift", linewidth=2)
plt.plot(t, omega_meas, "o-", label="measured Ω", alpha=0.55)
for name in ["joint_kalman", "robust_gated_kalman", "constrained_mpc", "cgcs_invariance_control"]:
    plt.plot(t, policies[name][0], label=name)
plt.title("Constraint-gated stability: Ω tracking")
plt.xlabel("calibration block")
plt.ylabel("Ω drift estimate / command")
plt.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_omega_tracking.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
plt.figure(figsize=(15, 6))
ax = plt.gca()
shade_windows(ax)
plt.plot(t, B_true, label="true B drift", linewidth=2)
plt.plot(t, B_meas, "o-", label="measured B", alpha=0.55)
for name in ["joint_kalman", "robust_gated_kalman", "constrained_mpc", "cgcs_invariance_control"]:
    plt.plot(t, policies[name][1], label=name)
plt.title("Constraint-gated stability: B tracking")
plt.xlabel("calibration block")
plt.ylabel("B drift estimate / command")
plt.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_b_tracking.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
candidate = "cgcs_invariance_control"
worst_block = int(np.argmax(rmse_by_policy[candidate]))

plt.figure(figsize=(14, 6))
plt.plot(pulse, target_curves[worst_block], label="target", linewidth=3)
for name in ["none", "moving_average", "joint_kalman", "robust_gated_kalman", "constrained_mpc", "cgcs_invariance_control", "oracle"]:
    plt.plot(pulse, policy_curves[name][worst_block], label=name)
plt.title(f"Constraint-gated stability: worst-case block {worst_block}")
plt.xlabel("time / pulse duration")
plt.ylabel("excited-state probability")
plt.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_worst_case_block_comparison.png", dpi=220, bbox_inches="tight")
plt.show()

print("Worst block for CGCS invariance control:", worst_block)


## 10. Minimal proof sketch

This notebook supports a compact stability argument:

1. If `cosθ ≥ threshold`, the update direction remains aligned with the target response.
2. Aligned updates reduce projection error rather than accumulating uncontrolled drift.
3. Misaligned updates are blocked or projected into a bounded recovery step.
4. Therefore, repeated CGCS-constrained updates remain bounded under this simulated disturbance model.

This is a proof sketch plus numerical evidence, not a formal theorem.


In [ ]:
# Save results

summary_csv = f"{RESULTS_DIR}/{NOTEBOOK_ID}_{SLUG}_policy_summary.csv"
invariance_csv = f"{RESULTS_DIR}/{NOTEBOOK_ID}_{SLUG}_invariance_summary.csv"
summary_json = f"{RESULTS_DIR}/{NOTEBOOK_ID}_{SLUG}_summary.json"

summary_df.to_csv(summary_csv, index=False)
invariance_df.to_csv(invariance_csv, index=False)

best_policy = str(summary_df.iloc[0]["policy"])
best_invariance_policy = str(invariance_df.iloc[0]["policy"])

summary_obj = {
    "notebook_id": NOTEBOOK_ID,
    "slug": SLUG,
    "title": TITLE,
    "threshold": float(THRESHOLD),
    "best_policy_by_mean_rmse": best_policy,
    "best_policy_by_invariance": best_invariance_policy,
    "noise_burst_windows": noise_burst_windows,
    "model_mismatch_windows": model_mismatch_windows,
    "cgcs_acceptance_rate": float(np.mean(cgcs_accept[1:])),
    "worst_block_cgcs_invariance_control": int(worst_block),
    "results": summary_df.to_dict(orient="records"),
    "invariance": invariance_df.to_dict(orient="records"),
}

with open(summary_json, "w") as f:
    json.dump(summary_obj, f, indent=2)

print("Saved:")
print(" ", summary_csv)
print(" ", invariance_csv)
print(" ", summary_json)


In [ ]:
# Export docs markdown with repo-relative image paths

figures = [
    ("CGCS phase-lock stability", "cgcs_stability_comparison", "Cosine similarity against target response makes the 45° phase-lock gate visible."),
    ("Threshold margin", "threshold_margin", "Margin is cosine similarity minus the 45° threshold."),
    ("Phase-error integral", "phase_error_integral", "Lower is better; integrates cosine-alignment error across the calibration stream."),
    ("Bounded RMSE radius", "bounded_rmse_radius", "95th percentile response RMSE summarizes bounded response error."),
    ("Update-gate trace", "update_gate_trace", "The CGCS controller adapts update radius without accepting unconstrained jumps."),
    ("Rejection events", "rejection_events", "Rejected update events show where the constraint prevents misaligned updates."),
    ("Policy ranking", "policy_ranking_summary", "Mean and max RMSE keep this notebook comparable to earlier control-stack notebooks."),
    ("Response RMSE comparison", "response_rmse_comparison", "Response-level RMSE shows block-by-block control quality."),
    ("Failure-event map", "failure_event_map", "A direct map of threshold violations by policy and calibration block."),
    ("Ω tracking", "omega_tracking", "Ω drift tracking under noise bursts and model mismatch."),
    ("B tracking", "b_tracking", "B drift tracking under coupled disturbance pressure."),
    ("Worst-case block comparison", "worst_case_block_comparison", "Worst-case response comparison for the CGCS invariance policy."),
]

doc_path = f"{DOCS_DIR}/{NOTEBOOK_ID}_{SLUG}.md"

metric_lines = "\n".join([
    f"| 45° threshold | {THRESHOLD:.6f} |",
    f"| best policy by mean RMSE | `{best_policy}` |",
    f"| best policy by invariance | `{best_invariance_policy}` |",
    f"| CGCS acceptance rate | {np.mean(cgcs_accept[1:]):.3f} |",
    f"| worst CGCS block | {worst_block} |",
])

figure_sections = "\n\n".join([
    f"### {title}\n\n![{title}](../figures/{SLUG}/{NOTEBOOK_ID}_{SLUG}_{fname}.png)\n\n{caption}"
    for title, fname, caption in figures
])

md_text = f"""# Notebook {NOTEBOOK_ID}: {TITLE}

Constraint-gated control defines a stability condition for adaptive updates:

```text
cosθ >= 1 / sqrt(1² + 1²)
```

When a proposed update falls below this threshold, the update is blocked or projected into a bounded recovery step.

## Metrics

| Metric | Value |
|---|---|
{metric_lines}

## Results files

- `results/{NOTEBOOK_ID}_{SLUG}_policy_summary.csv`
- `results/{NOTEBOOK_ID}_{SLUG}_invariance_summary.csv`
- `results/{NOTEBOOK_ID}_{SLUG}_summary.json`

## Figures

{figure_sections}

## Interpretation

Notebook {NOTEBOOK_ID} turns the control-stack experiments into a compact stability principle: CGCS gating keeps update propagation bounded by accepting only response-aligned updates.

```text
adaptive + unconstrained -> drift accumulation
adaptive + CGCS-constrained -> bounded update propagation
```
"""

with open(doc_path, "w") as f:
    f.write(md_text)

print("Saved docs markdown:", doc_path)


In [ ]:
# Zip Notebook 17 outputs for Colab download

zip_name = f"{NOTEBOOK_ID}_{SLUG}_outputs.zip"

files_to_zip = [
    f"{RESULTS_DIR}/{NOTEBOOK_ID}_{SLUG}_policy_summary.csv",
    f"{RESULTS_DIR}/{NOTEBOOK_ID}_{SLUG}_invariance_summary.csv",
    f"{RESULTS_DIR}/{NOTEBOOK_ID}_{SLUG}_summary.json",
    f"{DOCS_DIR}/{NOTEBOOK_ID}_{SLUG}.md",
]

for _, fname, _ in figures:
    files_to_zip.append(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_{fname}.png")

with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in files_to_zip:
        if os.path.exists(path):
            z.write(path)
        else:
            print("[warn] missing:", path)

print("Created:", zip_name)

try:
    from google.colab import files
    files.download(zip_name)
except Exception as e:
    print("Colab download skipped or unavailable:", e)
    print("Download the zip from the notebook file browser:", zip_name)
